# LLM-as-a-judge: subjective rewards, haiku edition

Remember the haiku example we did way back in
[000_rl_basics](../000_rl_basics/000_rl_basics.py)?
We trained Qwen3-4B to write haikus using a **verifiable reward** —
counting syllables to check the 5-7-5 structure. That works great
for format, but what about whether the poem is actually *good*?

What happens if we want to get an LLM involved?

This tutorial introduces **LLM-as-a-judge** — using a second,
larger model to score subjective quality (relevance, imagery,
poetic merit) alongside the deterministic structure score.
The combined reward teaches the model to write haikus that are
both well-formed *and* worth reading.

The flow:
1. Deploy the base model (Qwen3-4B) and a **judge model** (Qwen3-8B).
2. Define structure scoring (syllable counting, same as before).
3. Define an LLM judge prompt that scores haiku quality.
4. Build a combined reward: structure + LLM judge.
5. Evaluate the base model, train with GRPO, evaluate the trained model.
6. Compare.

## Prerequisites

This tutorial requires a Modal Secret named `huggingface-secret` containing your
`HF_TOKEN`. Create one at [modal.com/secrets](https://modal.com/secrets) if you
haven't already — the cell below fails fast with instructions otherwise.

> **Note:** you do **not** need to attach a GPU to this notebook. All training and
> serving happens on Modal-managed GPU workers spun up by the SDK — the notebook
> itself only needs to issue API calls.

In [ ]:
import modal

try:
    modal.Secret.from_name("huggingface-secret").hydrate()
except modal.exception.NotFoundError as e:
    raise RuntimeError(
        "Missing Modal Secret 'huggingface-secret'. Create one at "
        "https://modal.com/secrets with an HF_TOKEN entry, then re-run."
    ) from e

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main nltk

In [ ]:
import re

from modal_training_gym import (
    DeploymentConfig,
    EvalConfig,
    EvalRowResult,
    HuggingFaceDataset,
    Qwen3_4B,
    Qwen3_8B,
    SlimeRecipe,
    TrainConfig,
    list_checkpoints,
)

## Deploy the base model and the judge

We need two models running:
- **Qwen3-4B** — the model we're training (same as 000_rl_basics).
- **Qwen3-8B** — a larger model that acts as the judge. It never
  gets trained; it just scores haiku quality during rollouts.

Why a separate model? The judge needs to be better than the student
at recognizing good poetry. Using a bigger model from the same
family works well here.

In [ ]:
base_model = Qwen3_4B()
base_deployment = DeploymentConfig(model=base_model).serve()
print(f"Base model: {base_deployment.url}")

judge_model = Qwen3_8B()
judge_deployment = DeploymentConfig(
    model=judge_model,
    app_name="qwen3-8b-haiku-judge",
    served_model_name="qwen3-8b-judge",
).serve()
print(f"Judge model: {judge_deployment.url}")

Let's make sure both models are alive.

In [ ]:
response = base_deployment.generate(
    "Write a haiku about the ocean.",
    chat_template_kwargs={"enable_thinking": False},
)
print(f"Base model:\n{response}\n")

judge_response = judge_deployment.generate(
    "Rate this haiku on a scale of 1-10:\n\nWaves crash on the shore\nSalt and foam beneath the sky\nThe tide pulls away",
    chat_template_kwargs={"enable_thinking": False},
)
print(f"Judge model:\n{judge_response}")

## Structure scoring (recap)

This is the same syllable-counting logic from 000_rl_basics.
A perfect 5-7-5 haiku scores 0 (no deviation); each syllable
off target subtracts a point. We normalize this to [0, 1].

In [ ]:
_cmudict_cache = {}

def _get_cmudict() -> dict:
    if not _cmudict_cache:
        import nltk
        from nltk.corpus import cmudict
        nltk.download("cmudict", quiet=True)
        _cmudict_cache.update(cmudict.dict())
    return _cmudict_cache

def _count_syllables(text: str) -> int:
    cmu = _get_cmudict()
    total = 0
    for word in re.findall(r"[a-zA-Z]+", text):
        phones = cmu.get(word.lower())
        if phones:
            total += sum(p[-1].isdigit() for p in phones[0])
        else:
            count = len(re.findall(r"[aeiouy]+", word.lower()))
            if word.lower().endswith("e") and count > 1:
                count -= 1
            total += max(count, 1)
    return total

def score_structure(response: str) -> float:
    lines = [l.strip() for l in response.strip().split("\n") if l.strip()]
    if len(lines) != 3:
        return 0.0
    total_diff = sum(
        abs(_count_syllables(line) - target)
        for line, target in zip(lines, [5, 7, 5])
    )
    return max(0.0, 1.0 - total_diff / 10.0)

## The LLM judge

Here's the new part. We write a prompt that asks the judge model
to score two things:

- **Relevance** (0-5): does the haiku actually address the topic?
- **Poetic quality** (0-5): does it have imagery, emotion, a clear theme?

The judge outputs a single number. We parse it, normalize to [0, 1],
and combine it with the structure score.

`score_judge` takes the judge URL as a parameter and calls the
`/v1/chat/completions` endpoint with `aiohttp`. It runs inside
the training container during rollouts, so we install `aiohttp`
via `image_overlay` later.

In [ ]:
def build_judge_prompt(topic: str, haiku: str) -> str:
    return (
        "You are evaluating a haiku poem. Score the response on two criteria:\n"
        "\n"
        "Relevance (0-5 points):\n"
        f"- 5: the haiku's central theme and imagery directly evoke \"{topic}\"\n"
        f"- 3: the haiku mentions \"{topic}\" but it is not the central focus\n"
        f"- 1: the haiku is loosely related to \"{topic}\"\n"
        f"- 0: the haiku is not relevant to \"{topic}\"\n"
        "\n"
        "Poetic quality (0-5 points):\n"
        "- 5: vivid imagery, clear theme, emotional resonance\n"
        "- 3: makes sense and reads like a poem, but is unremarkable\n"
        "- 1: grammatically coherent but not poetic\n"
        "- 0: incoherent or not a poem\n"
        "\n"
        f"Topic: {topic}\n"
        "\n"
        "Haiku to evaluate:\n"
        f"{haiku}\n"
        "\n"
        "Output ONLY a single number (0-10), the sum of both scores. Nothing else."
    )

async def score_judge(judge_url: str, topic: str, haiku: str) -> float:
    import asyncio

    import aiohttp

    prompt = build_judge_prompt(topic, haiku)
    for attempt in range(5):
        try:
            async with aiohttp.ClientSession() as session:
                async with session.post(
                    f"{judge_url}/v1/chat/completions",
                    json={
                        "model": "qwen3-8b-judge",
                        "messages": [{"role": "user", "content": prompt}],
                        "max_tokens": 16,
                        "chat_template_kwargs": {"enable_thinking": False},
                    },
                    timeout=aiohttp.ClientTimeout(total=60),
                ) as resp:
                    if resp.status != 200:
                        return 0.0
                    data = await resp.json()
                    text = data["choices"][0]["message"]["content"].strip()
                    match = re.search(r"(\d+(?:\.\d+)?)", text)
                    if match:
                        score = float(match.group(1))
                        return min(max(score, 0), 10) / 10.0
                    return 0.0
        except (aiohttp.ClientError, asyncio.TimeoutError):
            if attempt < 4:
                await asyncio.sleep(2 ** attempt)
                continue
            return 0.0

Let's test the judge on a sample haiku to see what it returns.

In [ ]:
good_haiku = "Waves crash on the shore\nSalt and foam beneath the sky\nThe tide pulls away"
bad_haiku = "I like pizza lots\nPizza pizza pizza yum\nMore pizza for me"

good_score = await score_judge(judge_deployment.url, "ocean", good_haiku)
bad_score = await score_judge(judge_deployment.url, "ocean", bad_haiku)
print(f"Good haiku (on-topic): {good_score:.2f}")
print(f"Bad haiku (off-topic): {bad_score:.2f}")

## Combined reward: structure + judge

The reward function for training combines both signals:

- **Structure** (0-1): is the 5-7-5 syllable format correct?
- **Judge quality** (0-1): is the haiku relevant and poetic?

We weight them equally and scale the total to [-1, 1] so that
a mediocre haiku gets a negative reward and a good one gets
positive. The judge score is only evaluated when the structure
is reasonable (score > 0.5) — no point grading the poetry of
something that isn't even a haiku.

`make_haiku_judge_rm` is a factory that takes the judge URL
and returns the reward function. This lets us bind the URL
at runtime (after deploying the judge) while keeping the
function definition at module scope for cloudpickle.

In [ ]:
def make_haiku_judge_rm(judge_url: str):
    async def haiku_judge_rm(args, sample, **kwargs) -> float:
        response = sample.response
        structure = score_structure(response)

        quality = 0.0
        if structure > 0.5:
            topic = sample.prompt.split("about")[-1].strip().rstrip(".")
            quality = await score_judge(judge_url, topic, response)

        combined = structure + quality
        return combined - 1.0

    return haiku_judge_rm

## Dataset

Same haiku dataset as 000_rl_basics — topics from
[statworx/haiku](https://huggingface.co/datasets/statworx/haiku).

In [ ]:
class HaikuDataset(HuggingFaceDataset):
    hf_repo = "statworx/haiku"
    input_column = "keywords"
    output_column = "text"
    output_format = "jsonl"
    apply_chat_template = True
    system_prompt = (
        "You are a haiku poet. Write a haiku about the given topic. "
        "Use the 5-7-5 syllable format across three lines."
    )
    prompt_template = "Write a haiku about {input}."
    always_prepare = True

train_dataset = HaikuDataset(n_rows=10)
eval_dataset = HaikuDataset(n_rows=5)

## Evaluate the base model

Before training, let's see how the base model does.
We use the structure score for the eval metric — it's deterministic
and fast. The LLM judge adds cost per call, so we save it
for the reward signal during training.

In [ ]:
def eval_response_fn(_example: dict, response: str) -> EvalRowResult:
    return EvalRowResult(
        score=score_structure(response),
        response=response,
    )

eval_config = EvalConfig(
    dataset=eval_dataset,
    eval_response_fn=eval_response_fn,
    generate_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
)
print("——— Evaluating base model... ———")
base_eval = eval_config.evaluate(base_deployment, debug=True)
print(f"Base structure score: {base_eval.mean:.2f}")

## Train with the LLM judge reward

Now we bind the judge URL and train. The `custom_rm_function`
is cloudpickled into the training container — `judge_deployment.url`
is captured in the closure so the reward function can call the
judge during rollouts.

The `image_overlay` installs `aiohttp` and `nltk` in the training
container — the reward function needs both to call the judge
endpoint and count syllables.

In [ ]:
haiku_judge_rm = make_haiku_judge_rm(judge_deployment.url)

training_run = TrainConfig(
    model=base_model,
    dataset=train_dataset,
    recipe=SlimeRecipe(
        custom_rm_function=haiku_judge_rm,

        gpu_type="H100",
        colocate=True,
        tensor_model_parallel_size=1,
        sequence_parallel=False,
        rollout_num_gpus_per_engine=1,

        num_rollout=10,
        rollout_batch_size=16,
        rollout_max_response_len=4096,
        rollout_temperature=1.0,

        save_interval=5,
        apply_chat_template_kwargs='{"enable_thinking": false}',

        image_overlay=lambda image: image.run_commands(
            "uv pip install --system aiohttp nltk>=3.8.0",
            "python -c \"import nltk; nltk.download('cmudict', quiet=True)\"",
        ),
    ),
)
print("——— Starting training with LLM judge reward... ———")
train_result = training_run.train()
print("——— Training complete ———")

## Serve and evaluate the trained model

Let's serve the final checkpoint and run the same eval.

In [ ]:
checkpoint = list_checkpoints(train_result.training_run_id)[-1]
print(f"Checkpoint: {checkpoint.path}")

trained_deployment = DeploymentConfig(
    model=Qwen3_4B(),
    checkpoint=checkpoint,
    app_name="qwen3-4b-haiku-judge-serve",
    served_model_name="qwen3-4b-haiku-judge",
).serve()
print(f"Trained model: {trained_deployment.url}")

In [ ]:
print("——— Evaluating trained model... ———")
trained_eval = eval_config.evaluate(trained_deployment, debug=True)
print(f"Trained structure score: {trained_eval.mean:.2f}")

## Compare

The structure score tells us about format compliance. But the
real gain from the LLM judge is in subjective quality — the
trained model should produce haikus that are more relevant and
poetic, even if the structure score doesn't move much.

In [ ]:
print(f"Base structure score:    {base_eval.mean:.2f}")
print(f"Trained structure score: {trained_eval.mean:.2f}")
print(f"Delta:                   {trained_eval.mean - base_eval.mean:+.2f}")

## Next steps

- **Bigger judge**: swap `Qwen3_8B` for a 30B or 70B model for
  richer quality signals. Larger judges produce more nuanced scores
  but cost more per rollout.
- **Curriculum gating**: only enable the LLM judge after the model
  has learned basic structure (e.g. after N rollouts), so early
  training focuses on format and later training refines quality.
- **Multi-criteria scoring**: add more dimensions to the judge prompt
  (emotion, surprise, use of specific vocabulary) and weight them
  differently in the reward.